# מפגש 9 — מחברת הדגמה למורה: מהקשר החזותי ל־Pearson
חישוב ידני קודם, פלט צפוי אחר כך, ורק אז הרצה. דוגמת הנסיעות היא מומצאת; נתוני Penguins הם מאגר כיתתי אמיתי.

## סיפור: ארבע נסיעות, שני משתנים
כל שורה כוללת מדידות מרחק וזמן מאותה נסיעה. שערו אם המתאם חיובי, שלילי או אפס.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

trips_df = pd.DataFrame({'distance_km':[1,2,3,4], 'time_min':[10,30,20,40]})
display(trips_df)
assert trips_df.shape == (4,2)

## תרשים פיזור וזוגות
אין למיין x ו־y בנפרד. הנקודה (2,30) מתייחסת לאותה נסיעה.

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(trips_df['distance_km'],trips_df['time_min'])
plt.xlabel('Distance (km)')
plt.ylabel('Travel time (minutes)')
plt.title('Four illustrative trips')
plt.show()

## חישוב ידני מתועד
ממוצעים x=2.5 ק״מ, y=25 דקות. סכום מכפלות הסטיות 40, סכומי הריבועים 5 ו־500. לכן r=40/sqrt(2500)=0.8 ללא יחידות.

In [ ]:
x=trips_df['distance_km']
y=trips_df['time_min']
cross=((x-x.mean())*(y-y.mean())).sum()
ssx=((x-x.mean())**2).sum()
ssy=((y-y.mean())**2).sum()
manual_r=cross/np.sqrt(ssx*ssy)
print('Cross:',cross,'SSx:',ssx,'SSy:',ssy)
print('Hand formula:',manual_r,'Pandas:',x.corr(y))
assert cross==40 and ssx==5 and ssy==500
assert np.isclose(manual_r,0.8) and np.isclose(x.corr(y),0.8)

## שלוש צורות הקשר
Pearson מתאר קשר ליניארי. דוגמת U נותנת מתאם אפס אף ש־y=x² בדיוק.

In [ ]:
toy_x=pd.Series([-2.,-1.,0.,1.,2.])
toy_y=toy_x**2
print('Quadratic r:',toy_x.corr(toy_y))
plt.figure(figsize=(6,4))
plt.scatter(toy_x,toy_y)
plt.xlabel('x')
plt.ylabel('y = x squared')
plt.title('Exact nonlinear relationship; Pearson r = 0')
plt.show()
assert np.isclose(toy_x.corr(toy_y),0,atol=1e-12)

## מעבר לנתונים אמתיים
בחומר ההשתלמות נלמדות מטריצת מתאמים וקבוצת פיתוח. פה נדגים אותה חלוקה של תלמידים עם random_state=17. קבוצת heldout שמורה ואין משתמשים בה לבחירת קשר.

In [ ]:
from sklearn.model_selection import train_test_split
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
penguins_df=pd.read_csv(url)
development,heldout=train_test_split(penguins_df,test_size=0.25,random_state=17)
print('Source shape:',penguins_df.shape,'Development rows:',len(development))
assert penguins_df.shape==(344,7)
assert len(development)+len(heldout)==344

## זוגות ידועים, תרשים ומטריצה
המתאם מחושב רק על שורות שבהן שני השדות ידועים. המורה מדגים איך מפרידים מציאת קשר מהוכחת סיבתיות.

In [ ]:
import seaborn as sns
pair=development[['flipper_length_mm','body_mass_g']].dropna()
print('Known pairs:',len(pair),'Development r:',pair['flipper_length_mm'].corr(pair['body_mass_g']))
plt.figure(figsize=(6,4))
sns.scatterplot(data=pair,x='flipper_length_mm',y='body_mass_g')
plt.xlabel('Flipper length (mm)')
plt.ylabel('Body mass (g)')
plt.show()
numeric=['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
corr=development[numeric].corr()
plt.figure(figsize=(7,5))
sns.heatmap(corr,vmin=-1,vmax=1,annot=True,fmt='.2f')
plt.title('Development only: Pearson r')
plt.tight_layout()
plt.show()
assert 0 < len(pair) <= len(development)
assert np.isclose(corr.loc['flipper_length_mm','body_mass_g'],pair['flipper_length_mm'].corr(pair['body_mass_g']))

## דיון למורה
קשר חיובי אינו הוכחת סיבתיות. ממוצע או מתאם על כל הקובץ אינם בהכרח ערכי הפיתוח. העבירו לתלמידים את מעבדת ההשלמות שבה הם חוקרים זוג מדידות אחר.